#GPT2-XL Backbone Model

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"

# backbone model
BACKBONE = "openai-community/gpt2-xl"  # GPT2-XL

print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")

# ENTROPY SCORE COMPUTATION

def compute_entropy_score(text, model, tokenizer, device, max_len=512):

    try:
        with torch.no_grad():
            tokenized = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_len
            ).to(device)

            logits = model(**tokenized).logits[:, :-1]
            if logits.shape[1] == 0:
                return np.nan
            neg_entropy = F.softmax(logits, dim=-1) * F.log_softmax(logits, dim=-1)
            entropy = -neg_entropy.sum(-1).mean().item()

            return entropy

    except Exception as e:
        return np.nan

# PREPARE FUNCTION
def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER
def evaluate_entropy(texts, labels, model, tokenizer, device):

    scores = []

    print(f"Computing entropy for {len(texts)} samples...")

    for t in tqdm(texts, desc="Entropy Scoring", ncols=80):
        s = compute_entropy_score(t, model, tokenizer, device)
        scores.append(s)

    # Handle NaN values
    valid_scores = [s for s in scores if not (np.isnan(s) or np.isinf(s))]

    if not valid_scores:
        print(" No valid scores")
        return scores, [0]*len(labels), {
            'accuracy': 0.5,
            'auroc': 0.5,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'threshold': 0.0
        }

    median_score = np.median(valid_scores)
    scores = [median_score if (np.isnan(s) or np.isinf(s)) else s for s in scores]

    print(f"Valid scores: {len(valid_scores)}/{len(scores)}")
    print(f"Score range: [{min(scores):.4f}, {max(scores):.4f}]")
    auroc = roc_auc_score(labels, scores)
    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    print(f"Predictions: {preds.count(0)} label-0, {preds.count(1)} label-1")

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")


# DEFINE TASKS
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_entropy(X, y, model, tokenizer, device)
    res[name] = mt

    # Save results
    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'entropy_score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_entropy_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY

summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/entropy_{bn}_summary.csv")

print("\n" + "="*60)
print(f" FINAL SUMMARY")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results {PROJECT}/results/")


Using openai-community/gpt2-xl on cuda
Model loaded successfully!

Loading BEEMO dataset...
Loaded 2187 samples

H_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


Entropy Scoring: 100%|██████████████████████| 4374/4374 [04:04<00:00, 17.90it/s]


Valid scores: 4373/4374
Score range: [0.1896, 6.3828]
Predictions: 2198 label-0, 2176 label-1
AUROC: 0.3906 | Acc: 0.4159 | F1: 0.4144

E_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


Entropy Scoring: 100%|██████████████████████| 4374/4374 [04:21<00:00, 16.71it/s]


Valid scores: 4373/4374
Score range: [0.1896, 5.4023]
Predictions: 2191 label-0, 2183 label-1
AUROC: 0.4256 | Acc: 0.4449 | F1: 0.4444

H_vs_E
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


...

Valid scores: 8747/8748
Score range: [0.6772, 6.3828]
Predictions: 4386 label-0, 4362 label-1
AUROC: 0.5165 | Acc: 0.5138 | F1: 0.6106

G_vs_M
Samples: 8748 (6561/2187)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [06:43<00:00, 21.68it/s]


Valid scores: 8748/8748
Score range: [0.1896, 6.1641]
Predictions: 4380 label-0, 4368 label-1
AUROC: 0.4219 | Acc: 0.4532 | F1: 0.2703

H_vs_G
Samples: 8748 (2187/6561)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [06:22<00:00, 22.87it/s]


Valid scores: 8747/8748
Score range: [0.6377, 6.3828]
Predictions: 4387 label-0, 4361 label-1
AUROC: 0.4656 | Acc: 0.4842 | F1: 0.5869

 FINAL SUMMARY
         auroc  accuracy      f1
H_vs_M  0.3906    0.4159  0.4144
E_vs_M  0.4256    0.4449  0.4444
H_vs_E  0.4609    0.4684  0.4683
L_vs_M  0.3697    0.4280  0.2370
H_vs_L  0.5165    0.5138  0.6106
G_vs_M  0.4219    0.4532  0.2703
H_vs_G  0.4656    0.4842  0.5869

 Results /content/drive/MyDrive/BEEMO_Reproducibility/results/


#OPT-1.3B Backbone Model

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"

# backbone model
BACKBONE = "facebook/opt-1.3b"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")

# ENTROPY SCORE COMPUTATION

def compute_entropy_score(text, model, tokenizer, device, max_len=512):

    try:
        with torch.no_grad():
            tokenized = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_len
            ).to(device)

            logits = model(**tokenized).logits[:, :-1]
            if logits.shape[1] == 0:
                return np.nan
            neg_entropy = F.softmax(logits, dim=-1) * F.log_softmax(logits, dim=-1)
            entropy = -neg_entropy.sum(-1).mean().item()

            return entropy

    except Exception as e:
        return np.nan

# PREPARE FUNCTION
def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER
def evaluate_entropy(texts, labels, model, tokenizer, device):

    scores = []

    print(f"Computing entropy for {len(texts)} samples...")

    for t in tqdm(texts, desc="Entropy Scoring", ncols=80):
        s = compute_entropy_score(t, model, tokenizer, device)
        scores.append(s)

    # Handle NaN values
    valid_scores = [s for s in scores if not (np.isnan(s) or np.isinf(s))]

    if not valid_scores:
        print(" No valid scores")
        return scores, [0]*len(labels), {
            'accuracy': 0.5,
            'auroc': 0.5,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'threshold': 0.0
        }

    median_score = np.median(valid_scores)
    scores = [median_score if (np.isnan(s) or np.isinf(s)) else s for s in scores]

    print(f"Valid scores: {len(valid_scores)}/{len(scores)}")
    print(f"Score range: [{min(scores):.4f}, {max(scores):.4f}]")
    auroc = roc_auc_score(labels, scores)
    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    print(f"Predictions: {preds.count(0)} label-0, {preds.count(1)} label-1")

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")


# DEFINE TASKS
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_entropy(X, y, model, tokenizer, device)
    res[name] = mt

    # Save results
    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'entropy_score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_entropy_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY

summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/entropy_{bn}_summary.csv")

print("\n" + "="*60)
print(f" FINAL SUMMARY")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results {PROJECT}/results/")


Using facebook/opt-1.3b on cuda
Model loaded successfully!

Loading BEEMO dataset...
Loaded 2187 samples

H_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


Entropy Scoring: 100%|██████████████████████| 4374/4374 [02:32<00:00, 28.67it/s]


Valid scores: 4374/4374
Score range: [0.5186, 6.1172]
Predictions: 2187 label-0, 2187 label-1
AUROC: 0.3906 | Acc: 0.4252 | F1: 0.4252

E_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


Entropy Scoring: 100%|██████████████████████| 4374/4374 [02:49<00:00, 25.82it/s]


Valid scores: 4374/4374
Score range: [0.5186, 6.1172]
Predictions: 2189 label-0, 2185 label-1
AUROC: 0.4283 | Acc: 0.4513 | F1: 0.4511

H_vs_E
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


...

Valid scores: 8748/8748
Score range: [0.7773, 6.1172]
Predictions: 4387 label-0, 4361 label-1
AUROC: 0.5313 | Acc: 0.5251 | F1: 0.6197

G_vs_M
Samples: 8748 (6561/2187)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [04:23<00:00, 33.20it/s]


Valid scores: 8748/8748
Score range: [0.5186, 5.8711]
Predictions: 4378 label-0, 4370 label-1
AUROC: 0.4127 | Acc: 0.4516 | F1: 0.2684

H_vs_G
Samples: 8748 (2187/6561)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [04:06<00:00, 35.52it/s]


Valid scores: 8748/8748
Score range: [0.7388, 6.1172]
Predictions: 4383 label-0, 4365 label-1
AUROC: 0.4735 | Acc: 0.4911 | F1: 0.5925

 FINAL SUMMARY
         auroc  accuracy      f1
H_vs_M  0.3906    0.4252  0.4252
E_vs_M  0.4283    0.4513  0.4511
H_vs_E  0.4579    0.4698  0.4697
L_vs_M  0.3558    0.4227  0.2292
H_vs_L  0.5313    0.5251  0.6197
G_vs_M  0.4127    0.4516  0.2684
H_vs_G  0.4735    0.4911  0.5925

 Results /content/drive/MyDrive/BEEMO_Reproducibility/results/


#Falcon-7B Backbone Model

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"

# backbone model
BACKBONE = "tiiuae/falcon-7b"

print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")

# ENTROPY SCORE COMPUTATION

def compute_entropy_score(text, model, tokenizer, device, max_len=512):

    try:
        with torch.no_grad():
            tokenized = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_len
            ).to(device)

            logits = model(**tokenized).logits[:, :-1]
            if logits.shape[1] == 0:
                return np.nan
            neg_entropy = F.softmax(logits, dim=-1) * F.log_softmax(logits, dim=-1)
            entropy = -neg_entropy.sum(-1).mean().item()

            return entropy

    except Exception as e:
        return np.nan

# PREPARE FUNCTION
def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER
def evaluate_entropy(texts, labels, model, tokenizer, device):

    scores = []

    print(f"Computing entropy for {len(texts)} samples...")

    for t in tqdm(texts, desc="Entropy Scoring", ncols=80):
        s = compute_entropy_score(t, model, tokenizer, device)
        scores.append(s)

    # Handle NaN values
    valid_scores = [s for s in scores if not (np.isnan(s) or np.isinf(s))]

    if not valid_scores:
        print(" No valid scores")
        return scores, [0]*len(labels), {
            'accuracy': 0.5,
            'auroc': 0.5,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'threshold': 0.0
        }

    median_score = np.median(valid_scores)
    scores = [median_score if (np.isnan(s) or np.isinf(s)) else s for s in scores]
    auroc = roc_auc_score(labels, scores)
    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")


# DEFINE TASKS
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_entropy(X, y, model, tokenizer, device)
    res[name] = mt

    # Save results
    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'entropy_score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_entropy_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY

summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/entropy_{bn}_summary.csv")

print("\n" + "="*60)
print(f" FINAL SUMMARY")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results {PROJECT}/results/")


Using tiiuae/falcon-7b on cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!

Loading BEEMO dataset...
Loaded 2187 samples

H_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


Entropy Scoring: 100%|██████████████████████| 4374/4374 [03:51<00:00, 18.91it/s]


AUROC: 0.3926 | Acc: 0.4232 | F1: 0.4231

E_vs_M
Samples: 4374 (2187/2187)
Computing entropy for 4374 samples...


...

AUROC: 0.5342 | Acc: 0.5266 | F1: 0.6211

G_vs_M
Samples: 8748 (6561/2187)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [06:54<00:00, 21.10it/s]


AUROC: 0.4117 | Acc: 0.4544 | F1: 0.2723

H_vs_G
Samples: 8748 (2187/6561)
Computing entropy for 8748 samples...


Entropy Scoring: 100%|██████████████████████| 8748/8748 [06:42<00:00, 21.71it/s]


AUROC: 0.4794 | Acc: 0.4931 | F1: 0.5942

 FINAL SUMMARY
         auroc  accuracy      f1
H_vs_M  0.3926    0.4232  0.4231
E_vs_M  0.4191    0.4387  0.4383
H_vs_E  0.4691    0.4808  0.4800
L_vs_M  0.3620    0.4300  0.2392
H_vs_L  0.5342    0.5266  0.6211
G_vs_M  0.4117    0.4544  0.2723
H_vs_G  0.4794    0.4931  0.5942

 Results /content/drive/MyDrive/BEEMO_Reproducibility/results/
